# 14. Confirmation experiment: is JPEG compression the cause?

Notebook 13 rejected the resolution explanation (matching resolution closed 6% of the gap). It also showed, without a pre-specified criterion, that resizing alone did not cause the failure (2.6% at 112 px) while resizing plus JPEG quality 70 did (43.7%). This notebook tests that observation with new models, new images and criteria written before the run.

The criteria below were written on 12 November 2026 before any result. The values are in `CRIT` (next cell) and are recorded in `preregistration.json` with a timestamp before anything runs. The notebook stops if they are changed afterwards.

## Hypothesis

**H-JPEG.** The rejection failure (out-of-scope images labelled as disease) is caused by JPEG artefacts, and quality-aware augmentation repairs it through its JPEG component, not through resizing or noise.

## Part B: primary criteria

EfficientNet-B0, seeds 42/1/2, training code from notebook 13 (itself from 03/08). Metric: other→disease (%) on the original 888-image test set at **112 px + JPEG q70**. Gap = `base` − `qaug`.

| Arm | Training images (all classes) | Expected |
|---|---|---|
| `base` | no quality augmentation (reference, existing checkpoints) | high (about 44%) |
| `qaug` | resize p 0.6, JPEG p 0.6 (q 30–95), noise p 0.3 (reference, existing checkpoints) | low (about 9.5%) |
| `qaug_jpeg` | **JPEG only**, p 0.6, q 30–95 | low, near `qaug` |
| `qaug_resize` | **resize only**, p 0.6 | high, near `base` |
| `qaug_noise` | **noise only**, p 0.3 | high, near `base` |

- **P1** `qaug_jpeg` closes at least 80% of the gap and the 95% CI of (qaug_jpeg − base) excludes zero.
- **P2** `qaug_resize` closes less than 20%.
- **P3** `qaug_noise` closes less than 20%.
- **G** (validity) three seeds for every arm and a gap of at least 20 points. If G fails the result is *undecidable*, not *not confirmed*.

Verdict (exactly one): **confirmed** if P1, P2 and P3 hold. **Partially confirmed** if not, but `qaug_jpeg` closes at least 50% (CI excludes zero) and more than `qaug_resize` and `qaug_noise`, with the CI of (qaug_jpeg − qaug_resize) excluding zero. **Not confirmed** otherwise. The 80 px level is descriptive.

## Secondary criteria

| Code | Test | Criterion |
|---|---|---|
| **S1** reversal (`jpeg_other`: JPEG q 30–95 on every `other` image, p 1.0; disease images untouched) | compression artefacts are learned as class evidence and can be reversed | (a) closes at least 80% of the gap at 112 px; (b) other→disease on **original** images rises by at least 5 points over `base`, CI excludes zero; (c) disease→other at 112 px rises by at least 3 points, CI excludes zero |
| **S2** not general robustness (`jpeg_disease`: JPEG p 0.6 on disease images only) | JPEG on disease images alone does not repair rejection | closes less than 20% |
| **S3** orchard photographs (124 non-durian photographs of orchard O2, `base`, three seeds) | the JPEG effect also appears outside the public data | 112 px + JPEG q70 minus 112 px resize only at least 20 points, CI excludes zero. Without the orchard files: *not tested* |
| **S4** dose-response (`base`, 112 px, q in {95, 90, 80, 70, 50, 30}) | heavier compression, larger failure; light compression close to none | (a) q30 − q95 at least 20 points, CI excludes zero; (b) Spearman(q, other→disease) at most −0.8; (c) q95 − resize only below 10 points |

S4 uses the models and test images of notebook 13, so it is a finer measurement, not independent evidence.

## Part A: do the disease images carry more JPEG artefacts than the out-of-scope images?

Measure `lattice`: periodicity of block edges in the 256 px images (periods 6–60 px, cell 2). The measure must first pass a validity check on images with known compression:

- **V** on test images, AUROC for 112 px + JPEG versus original at least 0.70, and AUROC for 112 px resize only versus original at most 0.60. Otherwise part A is inconclusive.
- Wording on training images (AUROC of lattice for disease versus `other`, bootstrap CI): at least 0.60 with lower CI above 0.5 → "disease images carry more block artefacts"; at most 0.55 or CI covering 0.5 → "no evidence"; at most 0.45 with upper CI below 0.5 → "`other` carries more"; otherwise "small difference".

Part A is descriptive, not causal.

## Part C: quality grid (no training)

JPEG quality in {none, 95, 90, 80, 70, 50, 30} × size in {256, 224, 160, 112, 80} on every model with a checkpoint and on the orchard set. Used for S3, S4 and the grid figure.

## Running

1. Colab CPU: runs part A and builds the orchard cache from the Drive folder; part B is skipped without a GPU.
2. Kaggle GPU (T4, Internet on): attach the private dataset, the output of notebook 13 (for `base` seed 2) and the orchard cache if available. 15 training runs, then part C.
3. Results are written to `jpeg14_results.zip`.

In [ ]:
# 0. Settings and pre-specified criteria
import os, sys, io, re, json, time, math, random, shutil, zipfile, subprocess, warnings, datetime
from pathlib import Path
import numpy as np
import pandas as pd
from PIL import Image

T_START = time.time()
IN_KAGGLE = bool(os.environ.get('KAGGLE_KERNEL_RUN_TYPE'))
IN_COLAB = (not IN_KAGGLE) and Path('/var/colab/hostname').exists()
MOCK = os.environ.get('JPEG14_MOCK') == '1'      # synthetic data, for testing the code only

# ---------- Pre-specified criteria (12 Nov 2026). Do not edit after results are seen. ----------
CRIT = dict(
    level='112px',                       # level used for the verdict (112 px + JPEG q70)
    P1_jpeg_close_min=80.0,              # qaug_jpeg closes >= 80% of the gap and the CI of (qaug_jpeg - base) excludes 0
    P2_resize_close_max=20.0,            # qaug_resize closes < 20%
    P3_noise_close_max=20.0,             # qaug_noise closes < 20%
    G_min_gap=20.0, G_min_seeds=3,       # validity condition
    partial_jpeg_close_min=50.0,         # "partially confirmed"
    S1a_close_min=80.0, S1b_clean_rise_min=5.0, S1c_dis_as_other_rise_min=3.0,
    S2_close_max=20.0,
    S3_set='O2_A', S3_arm='base', S3_min_diff=20.0,                     # 112 + JPEG q70 minus 112 resize only
    S4_arm='base', S4_size=112, S4_min_range=20.0, S4_max_spearman=-0.8, S4_max_q95_minus_none=10.0,
    A_valid_auc_min=0.70, A_valid_resize_auc_max=0.60,                  # validity of the lattice measure
    A_more_auc=0.60, A_none_auc=0.55, A_less_auc=0.45,
)

RUN_PART_A = True          # CPU: compression artefacts by source
RUN_FIELD_BUILD = True     # CPU: build the orchard cache from clean_1024 (only if the Drive folder is present)
RUN_PART_B = True          # GPU: train the new arms; skipped without a GPU
RUN_PART_C = True          # JPEG quality x size grid (no training; reduced version without a GPU)
RUN_OPT_ARMS = True        # arms for the secondary criteria S1/S2

MODEL_NAME = 'efficientnet_b0.ra_in1k'
PRETRAINED = True
SEEDS = [42, 1, 2]
MAIN_ARMS = ['base', 'qaug', 'qaug_jpeg', 'qaug_resize', 'qaug_noise']
OPT_ARMS = ['jpeg_other', 'jpeg_disease']
ARMS = MAIN_ARMS + (OPT_ARMS if RUN_OPT_ARMS else [])
GRID_EXTRA_ARMS = ['qaug_other', 'qaug_disease', 'match_other', 'match_other_jpeg']   # arms from 13, evaluated in part C if checkpoints exist
# arm -> (quality-aug components, classes it applies to, JPEG probability)
ARM_SPEC = {
    'base': None,
    'qaug': dict(parts=('resize', 'jpeg', 'noise'), who='all'),
    'qaug_jpeg': dict(parts=('jpeg',), who='all'),
    'qaug_resize': dict(parts=('resize',), who='all'),
    'qaug_noise': dict(parts=('noise',), who='all'),
    'jpeg_other': dict(parts=('jpeg',), who='other', p_jpeg=1.0),
    'jpeg_disease': dict(parts=('jpeg',), who='disease'),
}
EPOCHS, PATIENCE, BATCH = 30, 8, 64
LR, WD, LABEL_SMOOTH = 1e-3, 0.05, 0.1
LEVELS, JPEG_Q = [224, 160, 112, 80], 70
IMG_SIZE = 224
# quality augmentation, same values as 03/08/13
Q_P_DOWNSCALE, Q_SCALE, Q_P_JPEG, Q_JPEG, Q_P_NOISE = 0.6, (0.25, 1.0), 0.6, (30, 96), 0.3
REUSE_CKPT = True          # reuse best.pt (08/03, output of 13, or an earlier run of 14) instead of retraining
SAVE_CKPT = True
N_BOOT, N_BOOT_FIELD = 2000, 5000
MAX_TRAIN_HOURS = 10.5     # no new runs after this, so the summary cells finish before Kaggle's 12 h limit
# part C
GRID_SIZES = [256, 224, 160, 112, 80]
GRID_Q_FULL = [None, 95, 90, 80, 70, 50, 30]      # None = no compression
GRID_Q_LITE = [None, 70]                          # used without a GPU
A_VALID_N = 300            # test images used to validate the part A measure

if MOCK:
    MODEL_NAME, PRETRAINED = 'mobilenetv3_small_050', False
    SEEDS, EPOCHS, PATIENCE, BATCH, N_BOOT, N_BOOT_FIELD = [42, 1], 2, 8, 16, 200, 300
    GRID_SIZES, A_VALID_N = [256, 112, 80], 40
    CRIT['G_min_seeds'] = 2
MODEL_SHORT = MODEL_NAME.split('.')[0]
print(f'platform: kaggle={IN_KAGGLE} colab={IN_COLAB} mock={MOCK}')

In [ ]:
# 1. Locate data (from 13), index existing checkpoints, record criteria
def _is_v11(p):
    s = str(p).lower()
    return ('v11' in s) or ('v1_1' in s)

def find_data_dir(root):
    c = [p.parent for p in Path(root).rglob('cache256.npy') if (p.parent / 'metadata.csv').exists()]
    if not c:
        raise FileNotFoundError(f'cache256.npy + metadata.csv not found under {root}')
    c.sort(key=lambda p: (_is_v11(p), str(p)))   # prefer v1
    return c[0]

if IN_KAGGLE:
    DATA_DIR = find_data_dir('/kaggle/input')
    OUT_DIR = Path('/kaggle/working/jpeg14')
    CACHE_PATH = DATA_DIR / 'cache256.npy'
    SEARCH_ROOTS = [Path('/kaggle/input')]
elif IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATA_DIR = Path('/content/drive/MyDrive/durian_work')
    OUT_DIR = DATA_DIR / 'results' / 'jpeg14'
    CACHE_PATH = Path('/content/cache256.npy')          # local copy; Drive reads drop out
    if not CACHE_PATH.exists():
        print('copying cache256.npy to local disk (1.2 GB)')
        shutil.copy(DATA_DIR / 'cache256.npy', CACHE_PATH)
    SEARCH_ROOTS = [DATA_DIR / 'results', DATA_DIR / 'field_cache', DATA_DIR / 'v1_1']
else:
    DATA_DIR = Path(os.environ.get('DURIAN_DATA_DIR', '.'))
    OUT_DIR = Path(os.environ.get('DURIAN_OUT_DIR', './jpeg14'))
    CACHE_PATH = DATA_DIR / 'cache256.npy'
    SEARCH_ROOTS = [DATA_DIR] + [Path(p) for p in os.environ.get('DURIAN_EXTRA_DIRS', '').split(os.pathsep) if p]
OUT_DIR.mkdir(parents=True, exist_ok=True)
SEARCH_ROOTS = [r for r in SEARCH_ROOTS if Path(r).exists()]

# Record the criteria before anything else runs. If a record exists and differs, the criteria were edited: stop.
PREREG = OUT_DIR / 'preregistration.json'
_prereg = dict(criteria=CRIT, arms=ARM_SPEC, main_arms=MAIN_ARMS, opt_arms=OPT_ARMS, seeds=SEEDS, model=MODEL_NAME,
               quality_aug=dict(p_downscale=Q_P_DOWNSCALE, scale=Q_SCALE, p_jpeg=Q_P_JPEG, jpeg_q=Q_JPEG, p_noise=Q_P_NOISE))
if PREREG.exists():
    _old = json.loads(PREREG.read_text())
    assert json.loads(json.dumps(_prereg['criteria'])) == _old['criteria'], 'CRIT differs from the recorded preregistration.json; criteria must not change after the first run'
else:
    PREREG.write_text(json.dumps(dict(written_utc=datetime.datetime.now(datetime.timezone.utc).isoformat(), **_prereg), indent=1, default=list))

meta_all = pd.read_csv(DATA_DIR / 'metadata.csv')
X = np.load(CACHE_PATH, mmap_mode='r')
assert len(meta_all) == X.shape[0], f'metadata has {len(meta_all)} rows but the cache has {X.shape[0]}'
assert X.shape[1:] == (256, 256, 3) and X.dtype == np.uint8, X.shape
meta_all['row'] = np.arange(len(meta_all))

# With the v1.1 dataset attached, orchard rows (group starting with orchard_) are kept out of training and used as a field test set.
is_orch = meta_all['group'].astype(str).str.startswith('orchard_') if 'group' in meta_all else pd.Series(False, index=meta_all.index)
meta = meta_all[~is_orch].reset_index(drop=True)
FIELD = {}                                               # set name -> uint8 N x 256 x 256 x 3 (non-durian orchard photographs)
if is_orch.any():
    FIELD['O1_A'] = np.stack([X[r] for r in meta_all.loc[is_orch, 'row']])
    print(f'{int(is_orch.sum())} orchard rows in metadata removed from training and kept as field set O1_A')
print('DATA_DIR =', DATA_DIR, '| OUT_DIR =', OUT_DIR, '| v1 images =', len(meta))
if not MOCK and len(meta) != 6051:
    print(f'WARNING: expected 6,051 v1 images, found {len(meta)}. Check the attached dataset.')

def load_classes():
    labels = sorted(meta['label'].unique())
    for p in sorted(Path(DATA_DIR).rglob('classes.json')):
        try:
            j = json.load(open(p, encoding='utf-8'))
            if isinstance(j, dict):
                j = j.get('classes', j)
            if isinstance(j, dict):
                if all(str(k).isdigit() for k in j):
                    j = [j[k] for k in sorted(j, key=int)]
                else:
                    j = [k for k, _ in sorted(j.items(), key=lambda kv: kv[1])]
            if sorted(j) == labels:
                return list(j)
        except Exception as e:
            print('could not read', p, ':', e)
    print('no matching classes.json; classes sorted alphabetically')
    return labels

CLASSES = load_classes()
C = len(CLASSES); OTHER = CLASSES.index('other')
meta['y'] = meta['label'].map({c: i for i, c in enumerate(CLASSES)}).astype(int)
meta['kind'] = np.where(meta['label'] == 'other', 'other', 'disease')
if 'source' not in meta: meta['source'] = meta['kind']
if 'group' not in meta: meta['group'] = meta['row'].astype(str)
SPLIT = {s: meta[meta['split'] == s] for s in ['train', 'val', 'test']}
IDX = {s: SPLIT[s]['row'].values for s in SPLIT}              # cache row indices (as IDX in 03/08)
y_all = np.full(len(meta_all), -1); y_all[meta['row'].values] = meta['y'].values
ROW_GROUP = dict(zip(meta['row'].values, meta['group'].astype(str).values))
print({s: len(d) for s, d in SPLIT.items()}, '| classes', C, '| other in train/test =',
      int((SPLIT['train'].kind == 'other').sum()), int((SPLIT['test'].kind == 'other').sum()))

# ---------- existing checkpoints (08/03 in the dataset, output of 13, earlier runs of 14) ----------
CKPT_INDEX = {}
for _root in SEARCH_ROOTS:
    for _p in sorted(Path(_root).rglob('best.pt')):
        if _is_v11(_p) or OUT_DIR in _p.parents: continue          # skip models trained on v1.1
        CKPT_INDEX.setdefault(_p.parent.name, []).append(_p.parent)

def find_ckpt(arm, seed):
    """Folder with best.pt for (arm, seed) trained on v1, or None. 03/08 naming first, then 13/14 naming."""
    names = ([f'{MODEL_SHORT}__{arm}__s{seed}'] if arm in ('base', 'qaug') else []) + [f'{arm}__s{seed}']
    for k, name in enumerate(names):
        for d in CKPT_INDEX.get(name, []):
            j = {}
            if (d / 'result.json').exists():
                try: j = json.loads((d / 'result.json').read_text())
                except Exception: j = {}
            if 'train_n' in j and int(j['train_n']) != len(IDX['train']): continue      # trained on other data
            if name == f'{arm}__s{seed}' and j.get('model') not in (None, MODEL_NAME): continue
            if name == f'{arm}__s{seed}' and j.get('arm') not in (None, arm): continue
            return d
    return None

_found = {f'{a}__s{s}': find_ckpt(a, s) for a in ARMS + GRID_EXTRA_ARMS for s in SEEDS}
print('reusable checkpoints:')
for k, v in _found.items():
    if v is not None: print(f'   {k:28s} ← {v}')
_miss_ref = [k for k in [f'{a}__s{s}' for a in ('base', 'qaug') for s in SEEDS] if _found[k] is None]
if _miss_ref: print('WARNING: reference checkpoints not found:', _miss_ref, '-> retrained in part B (base/qaug will differ slightly from 13). For base__s2, attach the output of notebook 13.')

In [ ]:
# 2. Quality measures and degradation (top part copied verbatim from 13), block-artefact measure, quality grid
GRID = np.array([0.125, 0.1875, 0.25, 0.3125, 0.375, 0.4375, 0.5, 0.625, 0.75, 0.875])
TAU, MIN_REF = 0.25, 1.0

def jpeg(im, q):
    b = io.BytesIO(); im.save(b, 'JPEG', quality=int(q)); b.seek(0)
    return Image.open(b).convert('RGB')

def eff_res(arr):
    """Effective resolution (0.125-1), copied from 13."""
    g = Image.fromarray(np.asarray(arr)).convert('L')
    g0 = np.asarray(g, np.float32)
    errs = np.empty(len(GRID), np.float32)
    for i, s in enumerate(GRID):
        n = max(8, int(round(256 * s)))
        errs[i] = np.abs(g0 - np.asarray(g.resize((n, n), Image.BICUBIC).resize((256, 256), Image.BICUBIC), np.float32)).mean()
    if errs[0] < MIN_REF:
        return np.nan
    c = errs / errs[0]
    below = np.where(c < TAU)[0]
    if len(below) == 0: return 1.0
    j = below[0]
    if j == 0: return float(GRID[0])
    return float(GRID[j-1] + (c[j-1] - TAU) / (c[j-1] - c[j] + 1e-9) * (GRID[j] - GRID[j-1]))

def lap_var(arr):
    g = np.asarray(Image.fromarray(np.asarray(arr)).convert('L'), np.float32)
    l = 4 * g[1:-1, 1:-1] - g[:-2, 1:-1] - g[2:, 1:-1] - g[1:-1, :-2] - g[1:-1, 2:]
    return float(l.var())

def degrade(a, s=160, q=70):                      # evaluation protocol, copied verbatim from 03/08
    im = Image.fromarray(a).resize((s, s), Image.BILINEAR)
    buf = io.BytesIO(); im.save(buf, 'JPEG', quality=q)
    return np.asarray(Image.open(buf).convert('RGB').resize((256, 256), Image.BICUBIC))

# standard levels (names as in 03/08), plus variants that separate JPEG from resizing (as in 13)
STD_LEVELS = ['original'] + [f'{L}px' for L in LEVELS]
VARIANTS = STD_LEVELS + ['jpeg_only'] + [f'{L}px_nojpeg' for L in LEVELS[1:]]

def apply_variant(a, name):
    a = np.asarray(a)
    if name == 'original': return a
    if name == 'jpeg_only': return np.asarray(jpeg(Image.fromarray(a), JPEG_Q))
    L = int(name.split('px')[0])
    if name.endswith('_nojpeg'):
        return np.asarray(Image.fromarray(a).resize((L, L), Image.BILINEAR).resize((256, 256), Image.BICUBIC))
    return degrade(a, L, JPEG_Q)

# ---------- Part C: size x JPEG quality grid (same functions as the protocol above) ----------
def cond_name(s, q):
    return f's{s}_q{"none" if q is None else q}'

def parse_cond(c):
    m = re.fullmatch(r's(\d+)_q(none|\d+)', c)
    return int(m.group(1)), (None if m.group(2) == 'none' else int(m.group(2)))

def apply_grid(a, s, q):
    a = np.asarray(a)
    if s >= 256:
        return a if q is None else np.asarray(jpeg(Image.fromarray(a), q))
    if q is None:
        return np.asarray(Image.fromarray(a).resize((s, s), Image.BILINEAR).resize((256, 256), Image.BICUBIC))
    return degrade(a, s, q)

# the grid must reproduce the variants of 13 pixel for pixel
_a = np.asarray(X[int(IDX['test'][0])])
for _v, (_s, _q) in {'original': (256, None), 'jpeg_only': (256, 70), '112px': (112, 70), '80px': (80, 70), '112px_nojpeg': (112, None), '160px': (160, 70)}.items():
    assert np.array_equal(apply_variant(_a, _v), apply_grid(_a, _s, _q)), _v

# ---------- Part A: block-artefact (lattice) measure ----------
# JPEG leaves seams at 8x8 (luma) and 16x16 (chroma) block edges. Averaging the size of neighbouring-pixel differences
# along columns (and rows) shows a periodic peak; for an image compressed small and enlarged, the period grows (112 px -> 18 px, 80 px -> 26 px).
# Score = highest periodogram peak for periods 6-60 px relative to the mean level around it (log10); independent of grid offset.
# Parameters (cap 4, lags 1-3, channels Y+Cb+Cr) were chosen on unrelated natural images before looking at the durian data.
L_NFFT, L_PMIN, L_PMAX, L_REL, L_CAP, L_LAGS = 2048, 6.0, 60.0, 0.35, 4.0, (1, 2, 3)
_LF = np.fft.rfftfreq(L_NFFT)
_LIDX = np.where((_LF >= 1 / L_PMAX) & (_LF <= 1 / L_PMIN))[0]
_LLO, _LHI = np.searchsorted(_LF, _LF[_LIDX] * (1 - L_REL)), np.searchsorted(_LF, _LF[_LIDX] * (1 + L_REL))

def lattice(arr):
    """Return (score, peak period in px). High score = periodic block-edge pattern. Flat image -> (nan, nan)."""
    ycc = np.asarray(Image.fromarray(np.asarray(arr)).convert('YCbCr'), np.float32)
    best, bp = -np.inf, np.nan
    for lag in L_LAGS:
        S, used = 0.0, 0
        for ch in range(3):
            g = ycc[..., ch]
            for gg in (g, g.T):
                dd = np.minimum(np.abs(gg[:, lag:] - gg[:, :-lag]), L_CAP).mean(0)
                dd = dd - dd.mean()
                P = np.abs(np.fft.rfft(dd * np.hanning(len(dd)), L_NFFT)) ** 2
                m = P[_LIDX].mean()
                if m > 1e-10: S = S + P / m; used += 1
        if not used: continue
        cs = np.r_[0.0, np.cumsum(S)]
        sc = S[_LIDX] / ((cs[_LHI] - cs[_LLO]) / (_LHI - _LLO) + 1e-12)
        j = int(sc.argmax())
        if sc[j] > 0 and math.log10(sc[j]) > best: best, bp = math.log10(sc[j]), 1 / _LF[_LIDX[j]]
    return (float(best), float(bp)) if np.isfinite(best) else (np.nan, np.nan)
print('functions ready | grid reproduces the variants of 13 pixel for pixel')

In [ ]:
# 2b. Orchard photographs (non-durian) for criterion S3
# Sources: (1) orchard_ rows in v1.1 metadata, if attached; (2) field_<name>_cache256.npy / orchard_cache256.npy files;
#          (3) built from clean_1024 on Drive. The resampling method is first checked against the existing O1 cache.
FIELD_SRC = {'O2_A': 'orchard_th/clean_1024/2569-11-10_O2/A'}             # sets to build
FIELD_CHECK = dict(npy='v1_1/orchard_cache256.npy', src='orchard_th/clean_1024/2569-11-10/A')   # O1: existing cache to check against
FIELD_EXPECT = {'O1_A': 59, 'O2_A': 124}
FIELD_TOL = 1.0            # mean absolute difference per pixel (0-255) accepted as "same resampling"
FIELD_FORCE = False        # True = build with bicubic even if the check fails (not recommended)
FIELD_DIR = DATA_DIR / 'field_cache'
_RES = {'bicubic': Image.BICUBIC, 'lanczos': Image.LANCZOS, 'bilinear': Image.BILINEAR, 'box': Image.BOX}
FIELD_METHODS = list(_RES) + ['crop_' + k for k in _RES]

def _list_images(d):
    return sorted(p for p in Path(d).iterdir() if p.suffix.lower() in ('.jpg', '.jpeg', '.png'))

def _open_rgb(p, tries=3):
    for t in range(tries):                                   # Drive reads fail now and then; retry
        try:
            with Image.open(p) as im: return im.convert('RGB')
        except OSError:
            if t == tries - 1: raise
            time.sleep(2)

def build256(im, how):
    if how.startswith('crop_'):
        w, h = im.size; s = min(w, h); l, t = (w - s) // 2, (h - s) // 2
        im, how = im.crop((l, t, l + s, t + s)), how[5:]
    return np.asarray(im.resize((256, 256), _RES[how]))

def choose_field_method():
    """Find the resampling that reproduces orchard_cache256.npy of O1 (built by notebook 11)."""
    npy, src = DATA_DIR / FIELD_CHECK['npy'], DATA_DIR / FIELD_CHECK['src']
    if not (npy.exists() and src.is_dir()):
        print(f'   cannot check resampling: {npy} or {src} not found'); return None, None
    ref = np.load(npy); ims = [_open_rgb(p) for p in _list_images(src)]
    if not len(ims): return None, None
    th = lambda a: np.asarray(Image.fromarray(a).convert('L').resize((16, 16), Image.BILINEAR), np.float32).ravel()
    tr, tf = np.stack([th(a) for a in ref]), np.stack([th(build256(im, 'bicubic')) for im in ims])
    match = np.abs(tr[:, None, :] - tf[None, :, :]).mean(2).argmin(1)          # cached image -> closest file
    k = np.linspace(0, len(ref) - 1, min(20, len(ref))).astype(int)
    diff = {how: float(np.mean([np.abs(ref[i].astype(np.float32) - build256(ims[match[i]], how)).mean() for i in k])) for how in FIELD_METHODS}
    best = min(diff, key=diff.get)
    print(f'   check against the O1 cache ({len(ref)} images, {len(ims)} files): mean difference per method =', {a: round(b, 2) for a, b in diff.items()})
    return (best if diff[best] <= FIELD_TOL else None), diff

if RUN_FIELD_BUILD and not IN_KAGGLE:
    for name, rel in FIELD_SRC.items():
        src, dst = DATA_DIR / rel, FIELD_DIR / f'field_{name}_cache256.npy'
        if dst.exists() or not src.is_dir(): continue
        print(f'building orchard cache {name} from {src}')
        how, diff = choose_field_method()
        if how is None and not FIELD_FORCE:
            print('   WARNING: resampling does not reproduce the existing cache (or nothing to check against); not built. S3 will be "not tested".'); continue
        how = how or 'bicubic'
        files = _list_images(src)
        arr = np.stack([build256(_open_rgb(p), how) for p in files])
        FIELD_DIR.mkdir(parents=True, exist_ok=True)
        np.save(dst, arr)
        pd.DataFrame(dict(idx=np.arange(len(files)), file=[p.name for p in files])).to_csv(FIELD_DIR / f'field_{name}_files.csv', index=False)
        (FIELD_DIR / f'field_{name}_build.json').write_text(json.dumps(dict(source=str(src), method=how, check_mean_abs_diff=diff, n=len(files), forced=bool(diff is None or diff.get(how, 9) > FIELD_TOL)), indent=1))
        print(f'   {dst.name}: {arr.shape} method {how}')
    _o1 = DATA_DIR / FIELD_CHECK['npy']
    if _o1.exists() and FIELD_DIR.exists() and not (FIELD_DIR / 'field_O1_A_cache256.npy').exists():
        shutil.copy(_o1, FIELD_DIR / 'field_O1_A_cache256.npy')         # keep both sets in one folder for a single Kaggle upload

for _root in SEARCH_ROOTS + [FIELD_DIR]:
    if not Path(_root).exists(): continue
    for _p in sorted(Path(_root).rglob('*cache256.npy')):
        _m = re.fullmatch(r'field_(.+)_cache256\.npy', _p.name)
        _name = _m.group(1) if _m else ('O1_A' if _p.name == 'orchard_cache256.npy' else None)
        if _name is None or _name in FIELD or '__' in _name: continue
        _a = np.load(_p)
        if _a.ndim == 4 and _a.shape[1:] == (256, 256, 3) and _a.dtype == np.uint8: FIELD[_name] = _a
print('orchard sets:', {k: len(v) for k, v in FIELD.items()} or 'none (S3 = not tested)')
for _k, _n in FIELD_EXPECT.items():
    if _k in FIELD and len(FIELD[_k]) != _n and not MOCK:
        print(f'WARNING: {_k} has {len(FIELD[_k])} images, expected {_n}. Check that the A folder excludes duplicates and X images.')
if CRIT['S3_set'] not in FIELD:
    print(f"no {CRIT['S3_set']} set: run this notebook on Colab first (it writes durian_work/field_cache/), then upload that folder as a Kaggle dataset")

In [ ]:
# 3. Part A: JPEG block artefacts by source (CPU)
from scipy.stats import spearmanr
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_val_predict, StratifiedKFold
from sklearn.metrics import roc_auc_score, f1_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler

def auc2(neg, pos):
    neg, pos = np.asarray(neg, float), np.asarray(pos, float)
    neg, pos = neg[~np.isnan(neg)], pos[~np.isnan(pos)]
    if len(neg) < 2 or len(pos) < 2: return np.nan
    return float(roc_auc_score(np.r_[np.zeros(len(neg)), np.ones(len(pos))], np.r_[neg, pos]))

def auc_ci(neg, pos, B=1000, seed=0):
    neg, pos = np.asarray(neg, float), np.asarray(pos, float)
    neg, pos = neg[~np.isnan(neg)], pos[~np.isnan(pos)]
    rng = np.random.default_rng(seed)
    bs = [auc2(rng.choice(neg, len(neg)), rng.choice(pos, len(pos))) for _ in range(B)]
    return auc2(neg, pos), float(np.percentile(bs, 2.5)), float(np.percentile(bs, 97.5))

A_RESULT = dict(status='not run')
if RUN_PART_A:
    # 3.1 measure every cached image (what the model sees)
    TFILE = OUT_DIR / 'trace_per_image.csv'
    if TFILE.exists() and len(pd.read_csv(TFILE)) == len(meta):
        tq = pd.read_csv(TFILE); print('using stored measurements', TFILE)
    else:
        t0 = time.time(); sc, pe = [], []
        for i, r in enumerate(meta['row']):
            s, p = lattice(X[r]); sc.append(s); pe.append(p)
            if (i + 1) % 1000 == 0: print(f'  lattice {i+1}/{len(meta)} ({time.time()-t0:.0f}s)')
        tq = meta[['row', 'label', 'kind', 'source', 'split']].copy()
        tq['lattice'], tq['lattice_period'] = sc, pe
        tq.to_csv(TFILE, index=False)
    meta['lattice'], meta['lattice_period'] = tq['lattice'].values, tq['lattice_period'].values

    # 3.2 validate the measure on test images with known compression (criterion V)
    samp = SPLIT['test'].sample(min(A_VALID_N, len(SPLIT['test'])), random_state=0)['row'].values
    vconds = [(v, (lambda a, v=v: apply_variant(a, v))) for v in VARIANTS if v != 'original'] + \
             [(cond_name(112, q), (lambda a, q=q: apply_grid(a, 112, q))) for q in (95, 90, 50, 30)]
    s0 = np.array([lattice(X[r])[0] for r in samp]); val = []
    for name, fn in vconds:
        r_ = [lattice(fn(np.asarray(X[r]))) for r in samp]; s1 = np.array([x[0] for x in r_])
        ok = ~np.isnan(s0) & ~np.isnan(s1)
        val.append(dict(condition=name, n=int(ok.sum()), median_original=float(np.nanmedian(s0)), median_condition=float(np.nanmedian(s1)),
                        auc_vs_original=auc2(s0, s1), paired_higher_pct=100 * float((s1[ok] > s0[ok]).mean()) if ok.any() else np.nan,
                        median_period_px=float(np.nanmedian([x[1] for x in r_]))))
    val = pd.DataFrame(val); val.to_csv(OUT_DIR / 'trace_validation.csv', index=False)
    print('\nMeasure validation on images of known compression (AUROC 0.5 = indistinguishable from the original):'); print(val.round(3).to_string(index=False))
    _g = lambda c: float(val.loc[val.condition == c, 'auc_vs_original'].iloc[0])
    A_VALID = bool(_g('112px') >= CRIT['A_valid_auc_min'] and _g('112px_nojpeg') <= CRIT['A_valid_resize_auc_max'])
    print(f"criterion V: AUROC 112px+JPEG = {_g('112px'):.3f} (needs >= {CRIT['A_valid_auc_min']}) | 112px resize only = {_g('112px_nojpeg'):.3f} (needs <= {CRIT['A_valid_resize_auc_max']}) -> " + ('met' if A_VALID else 'not met; part A inconclusive'))

    # 3.3 compare sources (training images = what the model learned from)
    tr = meta[meta['split'] == 'train']
    dis, oth = tr[tr.kind == 'disease'], tr[tr.kind == 'other']
    a, lo, hi = auc_ci(oth.lattice.values, dis.lattice.values)
    def _summ(d):
        return dict(n=len(d), lattice_median=d.lattice.median(), lattice_q25=d.lattice.quantile(.25), lattice_q75=d.lattice.quantile(.75),
                    period_median=d.lattice_period.median(), pct_nan=100 * d.lattice.isna().mean())
    by_src = pd.DataFrame([dict(kind=k, source=s, **_summ(d)) for (k, s), d in tr.groupby(['kind', 'source'])])
    by_cls = pd.DataFrame([dict(label=k, **_summ(d)) for k, d in tr.groupby('label')]).sort_values('lattice_median', ascending=False)
    by_src.to_csv(OUT_DIR / 'trace_by_source.csv', index=False); by_cls.to_csv(OUT_DIR / 'trace_by_class.csv', index=False)
    for col in ['lowres']:                                       # split by low-resolution flag, if present in metadata
        if col in tr:
            pd.DataFrame([dict(kind=k, **{col: v}, **_summ(d)) for (k, v), d in tr.groupby(['kind', col])]).to_csv(OUT_DIR / f'trace_by_{col}.csv', index=False)
    print('\nlattice by source (train):'); print(by_src.round(3).to_string(index=False))
    print('\nlattice by class (train):'); print(by_cls.round(3).to_string(index=False))

    # 3.4 does the lattice score add information beyond resolution and sharpness (eff_res, lap_var from 13)?
    qf = next((p for r_ in SEARCH_ROOTS + [OUT_DIR] for p in sorted(Path(r_).rglob('quality_per_image.csv'))), None)
    q13 = pd.read_csv(qf) if qf is not None else None
    if q13 is None or len(q13) != len(meta) or not np.array_equal(q13['row'].values, meta['row'].values):
        print('\nno matching quality_per_image.csv from 13; measuring eff_res / lap_var again (a few minutes)')
        q13 = meta[['row']].copy(); q13['eff_res'] = [eff_res(X[r]) for r in meta['row']]; q13['lap_var'] = [lap_var(X[r]) for r in meta['row']]
        q13.to_csv(OUT_DIR / 'quality_per_image.csv', index=False)
    meta['eff_res'], meta['lap_var'] = q13['eff_res'].values, q13['lap_var'].values
    tr = meta[meta['split'] == 'train']
    f = np.c_[tr.eff_res, np.log1p(tr.lap_var), tr.lattice]; f = np.where(np.isnan(f), np.nanmedian(f, 0), f)
    yb = (tr.kind == 'other').astype(int).values
    cv = StratifiedKFold(5, shuffle=True, random_state=0); auc_feat = {}
    for name, cols in [('eff_res+lap_var', [0, 1]), ('lattice', [2]), ('eff_res+lap_var+lattice', [0, 1, 2])]:
        p = cross_val_predict(make_pipeline(StandardScaler(), LogisticRegression(class_weight='balanced', max_iter=1000)), f[:, cols], yb, cv=cv, method='predict_proba')[:, 1]
        auc_feat[name] = float(roc_auc_score(yb, p))
    rho = spearmanr(tr.eff_res, tr.lattice, nan_policy='omit').statistic
    print('\nAUROC disease vs other (logistic, 5-fold):', {k: round(v, 3) for k, v in auc_feat.items()}, '| Spearman(eff_res, lattice) =', round(float(rho), 3))

    # 3.5 pre-specified wording
    if not A_VALID: word = 'inconclusive (measure failed criterion V)'
    elif a >= CRIT['A_more_auc'] and lo > 0.5: word = 'disease images carry more block artefacts than other (consistent with explanation a)'
    elif a <= CRIT['A_less_auc'] and hi < 0.5: word = 'other images carry more block artefacts than disease images (contradicts explanation a)'
    elif a <= CRIT['A_none_auc'] or lo <= 0.5 <= hi: word = 'no evidence that disease images carry more block artefacts than other'
    else: word = 'small difference'
    A_RESULT = dict(status='done', metric_valid=A_VALID, auc_disease_vs_other=a, ci_lo=lo, ci_hi=hi, wording=word,
                    median_disease=float(dis.lattice.median()), median_other=float(oth.lattice.median()),
                    validation={r_['condition']: r_['auc_vs_original'] for _, r_ in val.iterrows()}, auc_features=auc_feat, spearman_eff_lattice=float(rho))
    (OUT_DIR / 'trace_result.json').write_text(json.dumps(A_RESULT, indent=1, ensure_ascii=False))
    print(f'\n===== Part A: lattice AUROC (disease > other) = {a:.3f} [{lo:.3f}, {hi:.3f}] | median disease {dis.lattice.median():.3f}, other {oth.lattice.median():.3f}')
    print('pre-specified wording:', word)

In [ ]:
# 3b. Part A figures
import matplotlib
matplotlib.use('Agg') if not (IN_COLAB or IN_KAGGLE) else None
import matplotlib.pyplot as plt

def ecdf(ax, v, **k):
    v = np.asarray(v, float); v = np.sort(v[~np.isnan(v)])
    if len(v): ax.step(v, np.arange(1, len(v) + 1) / len(v), where='post', **k)

if RUN_PART_A:
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for (k, s), d in meta[meta['split'] == 'train'].groupby(['kind', 'source']):
        ecdf(ax[0], d.lattice.values, label=f'{k}: {s} (n={len(d)})', lw=2 if k == 'disease' else 1.3, ls='-' if k == 'disease' else '--')
    for _, r in val[val.condition.isin(['jpeg_only', '224px', '112px', '80px', '112px_nojpeg'])].iterrows():
        ax[0].axvline(r.median_condition, color='gray', lw=0.6, ls=':'); ax[0].text(r.median_condition, 1.02, r.condition, rotation=90, fontsize=7, ha='center', va='bottom')
    ax[0].axvline(val.median_original.iloc[0], color='k', lw=0.6, ls=':'); ax[0].text(val.median_original.iloc[0], 1.02, 'test original', rotation=90, fontsize=7, ha='center', va='bottom')
    ax[0].set(xlabel='block-lattice score (log10 peak prominence)', ylabel='cumulative fraction of training images', title='(a) Compression trace by source')
    ax[0].legend(fontsize=7, loc='lower right')
    v2 = val.set_index('condition')
    ax[1].barh(range(len(v2)), v2.auc_vs_original.values, color=['C3' if 'nojpeg' in c else 'C0' for c in v2.index])
    ax[1].axvline(0.5, color='k', lw=0.8); ax[1].set_yticks(range(len(v2))); ax[1].set_yticklabels(v2.index, fontsize=8)
    ax[1].set(xlabel='AUROC vs. original test images', title='(b) Metric validation on known compression', xlim=(0.3, 1.0))
    plt.tight_layout()
    for ext in ['png', 'pdf']: fig.savefig(OUT_DIR / f'fig_trace_by_source.{ext}', dpi=300)
    plt.show() if (IN_COLAB or IN_KAGGLE) else plt.close(fig)
    print('part A figures saved')

In [ ]:
# 4. Part B: training setup (from 13, itself from 03/08; TrainDS selects quality-aug components)
try:
    import torch
    HAS_GPU = torch.cuda.is_available()
except ImportError:
    HAS_GPU = False
DO_B = RUN_PART_B and (HAS_GPU or MOCK)
DO_C = RUN_PART_C
if RUN_PART_B and not DO_B:
    print('no GPU: part B skipped (part A is done). Run part B on a Kaggle GPU.')

if DO_B or DO_C:
    try:
        import timm
    except ImportError:
        subprocess.run([sys.executable, '-m', 'pip', '-q', 'install', 'timm'], check=False); import timm
    import torchvision.transforms.v2 as T
    from torch.utils.data import Dataset, DataLoader
    from sklearn.metrics import accuracy_score
    dev = 'cuda' if HAS_GPU else 'cpu'
    MEAN, STD = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)
    NUM_WORKERS = 0 if MOCK else (4 if IN_KAGGLE else 2)
    XR = np.load(CACHE_PATH) if DO_B else X                     # training: whole cache in RAM; evaluation only: memory-mapped

    def rand_quality(a, rng):                                   # copied verbatim from 03/08/13 (used by the qaug arm)
        if rng.random() < Q_P_DOWNSCALE:
            s = max(16, int(256 * rng.uniform(*Q_SCALE)))
            im = Image.fromarray(a).resize((s, s), Image.BILINEAR)
            a = np.asarray(im.resize((256, 256), rng.choice([Image.BILINEAR, Image.BICUBIC, Image.NEAREST])))
        if rng.random() < Q_P_JPEG:
            buf = io.BytesIO(); Image.fromarray(a).save(buf, 'JPEG', quality=int(rng.integers(*Q_JPEG)))
            a = np.asarray(Image.open(buf).convert('RGB'))
        if rng.random() < Q_P_NOISE:
            a = np.clip(a.astype(np.float32) + rng.normal(0, rng.uniform(2, 8), a.shape), 0, 255).astype(np.uint8)
        return a

    def rand_quality_parts(a, rng, parts, p_jpeg=Q_P_JPEG):     # identical to rand_quality, plus a switch per component
        if 'resize' in parts and rng.random() < Q_P_DOWNSCALE:
            s = max(16, int(256 * rng.uniform(*Q_SCALE)))
            im = Image.fromarray(a).resize((s, s), Image.BILINEAR)
            a = np.asarray(im.resize((256, 256), rng.choice([Image.BILINEAR, Image.BICUBIC, Image.NEAREST])))
        if 'jpeg' in parts and rng.random() < p_jpeg:
            buf = io.BytesIO(); Image.fromarray(a).save(buf, 'JPEG', quality=int(rng.integers(*Q_JPEG)))
            a = np.asarray(Image.open(buf).convert('RGB'))
        if 'noise' in parts and rng.random() < Q_P_NOISE:
            a = np.clip(a.astype(np.float32) + rng.normal(0, rng.uniform(2, 8), a.shape), 0, 255).astype(np.uint8)
        return a

    # With all components on, the output must equal rand_quality pixel for pixel (same seed); with one on, only that change may occur.
    _a0 = np.clip(np.asarray(X[int(IDX['train'][0])]).astype(np.int16) + np.random.default_rng(0).integers(-6, 7, (256, 256, 3)), 0, 255).astype(np.uint8)   # add texture so the image is not flat
    for _s in range(40):
        assert np.array_equal(rand_quality(_a0, np.random.default_rng(_s)), rand_quality_parts(_a0, np.random.default_rng(_s), ('resize', 'jpeg', 'noise'))), _s
    assert all(np.array_equal(rand_quality_parts(_a0, np.random.default_rng(_s), ()), _a0) for _s in range(5))
    _n = [not np.array_equal(rand_quality_parts(_a0, np.random.default_rng(_s), ('jpeg',), 1.0), _a0) for _s in range(20)]
    assert all(_n), 'jpeg with p=1.0 must change every image'
    _fr = {p: np.mean([not np.array_equal(rand_quality_parts(_a0, np.random.default_rng(_s), (p,)), _a0) for _s in range(1000)]) for p in ('resize', 'jpeg', 'noise')}
    assert abs(_fr['resize'] - Q_P_DOWNSCALE) < 0.06 and abs(_fr['jpeg'] - Q_P_JPEG) < 0.06 and abs(_fr['noise'] - Q_P_NOISE) < 0.06, _fr
    print('rand_quality_parts matches rand_quality | share of images changed with one component on (expected 0.6/0.6/0.3):', {k: round(float(v), 2) for k, v in _fr.items()})

    tf_train = T.Compose([
        T.RandomResizedCrop(IMG_SIZE, scale=(0.5, 1.0), antialias=True),
        T.RandomHorizontalFlip(), T.RandomVerticalFlip(), T.RandomRotation(20),
        T.ColorJitter(0.35, 0.35, 0.25, 0.02),
        T.RandomApply([T.GaussianBlur(5, (0.1, 2.0))], p=0.2),
        T.ToDtype(torch.float32, scale=True), T.Normalize(MEAN, STD)])
    tf_eval = T.Compose([T.Resize((IMG_SIZE, IMG_SIZE), antialias=True),
                         T.ToDtype(torch.float32, scale=True), T.Normalize(MEAN, STD)])

    def arm_applies(spec, y):
        return spec is not None and (spec['who'] == 'all' or (spec['who'] == 'other' and y == OTHER) or (spec['who'] == 'disease' and y != OTHER))

    class TrainDS(Dataset):
        def __init__(self, idx, arm):
            self.idx, self.arm, self.spec = np.asarray(idx), arm, ARM_SPEC[arm]
        def __len__(self): return len(self.idx)
        def __getitem__(self, k):
            i = int(self.idx[k]); y = int(y_all[i])
            rng = np.random.default_rng(torch.randint(0, 2**31 - 1, (1,)).item())
            a = XR[i]
            if self.arm == 'qaug': a = rand_quality(a, rng)                      # original path of 03/08/13
            elif arm_applies(self.spec, y): a = rand_quality_parts(a, rng, self.spec['parts'], self.spec.get('p_jpeg', Q_P_JPEG))
            return tf_train(torch.from_numpy(np.array(a)).permute(2, 0, 1)), y

    @torch.no_grad()
    def logits_of(model, arrs):
        model.eval(); out = []
        for s in range(0, len(arrs), BATCH):
            x = torch.stack([tf_eval(torch.from_numpy(np.array(a)).permute(2, 0, 1)) for a in arrs[s:s + BATCH]])
            with torch.autocast(dev, enabled=(dev == 'cuda')):
                out.append(model(x.to(dev, non_blocking=True)).float().cpu())
        return torch.cat(out)

    @torch.no_grad()
    def preds_multi(models, arrs):
        """Like logits_of, but preprocesses once for several models. Returns {name: predicted class} (used in part C)."""
        out = {t: [] for t in models}
        for s in range(0, len(arrs), BATCH):
            x = torch.stack([tf_eval(torch.from_numpy(np.array(a)).permute(2, 0, 1)) for a in arrs[s:s + BATCH]]).to(dev, non_blocking=True)
            with torch.autocast(dev, enabled=(dev == 'cuda')):
                for t, m in models.items(): out[t].append(m(x).float().argmax(1).cpu())
        return {t: torch.cat(v).numpy().astype(np.uint8) for t, v in out.items()}

    def ece(probs, y, n_bins=15):
        conf, pred = probs.max(1), probs.argmax(1); acc = (pred == y); e = 0.0
        edges = np.linspace(0, 1, n_bins + 1)
        for lo, hi in zip(edges[:-1], edges[1:]):
            m = (conf > lo) & (conf <= hi)
            if m.any(): e += m.mean() * abs(acc[m].mean() - conf[m].mean())
        return float(e)

    def fit_temperature(logits, y):
        logT = torch.zeros(1, requires_grad=True)
        opt = torch.optim.LBFGS([logT], lr=0.1, max_iter=200); yt = torch.as_tensor(y)
        def closure():
            opt.zero_grad(); loss = torch.nn.functional.cross_entropy(logits / logT.exp(), yt); loss.backward(); return loss
        opt.step(closure)
        return float(logT.detach().exp())

    def metrics(p, y):
        pred = p.argmax(1); o = y == OTHER
        return dict(acc=100 * accuracy_score(y, pred), macro_f1=100 * f1_score(y, pred, average='macro'),
                    other_as_disease=100 * float((pred[o] != OTHER).mean()), disease_as_other=100 * float((pred[~o] == OTHER).mean()),
                    acc_disease=100 * float((pred[~o] == y[~o]).mean()), ece=100 * ece(p, y))

    def make_model():
        return timm.create_model(MODEL_NAME, pretrained=PRETRAINED, num_classes=C, drop_rate=0.2).to(dev)

    if DO_B:
        t0 = time.time()
        TEST_ARRS = {v: [apply_variant(XR[i], v) for i in IDX['test']] for v in VARIANTS}
        VAL_ARRS = [XR[i] for i in IDX['val']]; VAL_DEG = [degrade(XR[i], 112) for i in IDX['val']]
        print(f'{len(VARIANTS)} evaluation variants prepared ({time.time()-t0:.0f}s) | device {dev}')

In [ ]:
# 5. Train and evaluate (from 13; checkpoint reuse extended to outputs of 13 and earlier runs of 14)
def evaluate_and_save(model, out, arm, seed, info):
    yt = y_all[IDX['test']]
    T_ = fit_temperature(logits_of(model, VAL_ARRS), y_all[IDX['val']])
    rob, probs = [], {}
    for v in VARIANTS:
        p = (logits_of(model, TEST_ARRS[v]) / T_).softmax(1).numpy(); probs[v] = p.astype(np.float32)
        rob.append(dict(variant=v, **metrics(p, yt)))
    pd.DataFrame(rob).to_csv(out / 'robustness.csv', index=False)
    np.savez_compressed(out / 'test_probs.npz', idx=IDX['test'], y=yt, **probs)
    (out / 'result.json').write_text(json.dumps(dict(arm=arm, seed=seed, model=MODEL_NAME, temperature=round(T_, 3), train_n=int(len(IDX['train'])),
        torch_version=torch.__version__, timm_version=timm.__version__, gpu=(torch.cuda.get_device_name(0) if HAS_GPU else 'cpu'), robustness=rob, **info), indent=1))
    r = {d['variant']: d for d in rob}
    print(f'{out.name} [{info["source"]}]: acc {r["original"]["acc"]:.2f} | other->disease original/112/80 = '
          f'{r["original"]["other_as_disease"]:.1f}/{r["112px"]["other_as_disease"]:.1f}/{r["80px"]["other_as_disease"]:.1f}')
    return r

def old_numbers(ref):
    """Read a stored robustness.csv -> {level: (acc %, other->disease %)}. 03/08 format (column level, fractions) or 13/14 (column variant, %)."""
    f = ref / 'robustness.csv'
    if not f.exists(): return {}
    d = pd.read_csv(f)
    if 'level' in d.columns: return {str(r['level']): (100 * r['acc'], 100 * r['other_as_disease']) for _, r in d.iterrows()}
    if 'variant' in d.columns: return {str(r['variant']): (r['acc'], r['other_as_disease']) for _, r in d.iterrows()}
    return {}

def reuse_checkpoint(arm, seed, out):
    # Models trained earlier: evaluate the stored best.pt with this code and check that it reproduces the stored numbers.
    if not REUSE_CKPT: return False
    ref = find_ckpt(arm, seed)
    if ref is None: return False
    model = make_model(); model.load_state_dict(torch.load(ref / 'best.pt', map_location=dev))
    r = evaluate_and_save(model, out, arm, seed, dict(source='reused_checkpoint', ref_dir=str(ref)))
    old = old_numbers(ref); lv = [l for l in STD_LEVELS if l in old]
    j = json.loads((out / 'result.json').read_text())
    if lv:
        d_acc = max(abs(r[l]['acc'] - old[l][0]) for l in lv); d_oad = max(abs(r[l]['other_as_disease'] - old[l][1]) for l in lv)
        ok = d_acc <= 0.35 and d_oad <= 2.7           # about 3 of 888 images / 2 of 77 (GPU and AMP nondeterminism)
        print(f'   check against stored results ({ref}): max diff acc {d_acc:.2f} pts, other->disease {d_oad:.2f} pts -> ' + ('match' if ok else 'MISMATCH: evaluation differs from training run, stop here'))
        j['ref_check'] = dict(max_abs_acc=d_acc, max_abs_other_as_disease=d_oad, ok=bool(ok))
    else:
        print(f'   WARNING: no stored robustness.csv in {ref} to check against'); j['ref_check'] = dict(ok=None)
    (out / 'result.json').write_text(json.dumps(j, indent=1))
    return True

def train_one(arm, seed):
    out = OUT_DIR / f'{arm}__s{seed}'
    if (out / 'result.json').exists():
        print(f'skip {out.name} (already done)'); return
    out.mkdir(parents=True, exist_ok=True)
    if reuse_checkpoint(arm, seed, out): return
    if (time.time() - T_START) / 3600 > MAX_TRAIN_HOURS:
        print(f'over {MAX_TRAIN_HOURS} h: {out.name} not started (rerun with this output attached as input to train only what is missing)'); out.rmdir(); return
    torch.manual_seed(seed); np.random.seed(seed); random.seed(seed)
    tr = IDX['train']
    counts = np.bincount(y_all[tr], minlength=C)
    w = torch.tensor(len(tr) / (C * counts), dtype=torch.float32, device=dev)
    crit = torch.nn.CrossEntropyLoss(weight=w, label_smoothing=LABEL_SMOOTH)
    model = make_model()
    dl = DataLoader(TrainDS(tr, arm), batch_size=BATCH, shuffle=True, num_workers=NUM_WORKERS,
                    pin_memory=(dev == 'cuda'), drop_last=True, persistent_workers=NUM_WORKERS > 0)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)
    sched = torch.optim.lr_scheduler.OneCycleLR(opt, max_lr=LR, total_steps=EPOCHS * len(dl), pct_start=0.1)
    scaler = torch.amp.GradScaler(enabled=(dev == 'cuda'))
    best, best_ep, hist, t0 = -1, 0, [], time.time()
    yv = y_all[IDX['val']]
    for ep in range(EPOCHS):
        model.train(); tot = 0
        for x, yb in dl:
            x, yb = x.to(dev, non_blocking=True), yb.to(dev, non_blocking=True)
            with torch.autocast(dev, enabled=(dev == 'cuda')):
                loss = crit(model(x), yb)
            opt.zero_grad(set_to_none=True); scaler.scale(loss).backward(); scaler.step(opt); scaler.update(); sched.step()
            tot += loss.item() * len(yb)
        f_clean = f1_score(yv, logits_of(model, VAL_ARRS).argmax(1).numpy(), average='macro')
        f_deg = f1_score(yv, logits_of(model, VAL_DEG).argmax(1).numpy(), average='macro')
        score = (f_clean + f_deg) / 2
        hist.append(dict(epoch=ep + 1, train_loss=tot / len(dl.dataset), val_f1_clean=f_clean, val_f1_112px=f_deg))
        print(f'  [{arm} s{seed}] ep {ep + 1:2d} | loss {hist[-1]["train_loss"]:.3f} | val F1 clean {f_clean:.4f} | 112px {f_deg:.4f}')
        if score > best:
            best, best_ep = score, ep + 1; torch.save(model.state_dict(), out / 'best.pt')
        elif ep + 1 - best_ep >= PATIENCE:
            print('  early stop'); break
    del dl
    model.load_state_dict(torch.load(out / 'best.pt', map_location=dev))
    pd.DataFrame(hist).to_csv(out / 'history.csv', index=False)
    evaluate_and_save(model, out, arm, seed, dict(source='trained', best_epoch=best_ep, train_minutes=round((time.time() - t0) / 60, 1)))
    if not SAVE_CKPT: (out / 'best.pt').unlink()

if DO_B:
    todo = [(a, s) for s in SEEDS for a in MAIN_ARMS] + [(a, s) for s in SEEDS for a in ARMS if a not in MAIN_ARMS]   # primary arms for all seeds first, then secondary arms
    print(f'{len(todo)} runs: {ARMS} x seed {SEEDS}')
    for a, s in todo:
        train_one(a, s)

In [ ]:
# 6. Part C: size x JPEG quality grid on every model with a checkpoint (no training)
GRID_DIR = OUT_DIR / 'grid'

def ckpt_path(arm, seed):
    p = OUT_DIR / f'{arm}__s{seed}' / 'best.pt'
    if p.exists(): return p
    d = find_ckpt(arm, seed)
    return (d / 'best.pt') if d is not None else None

if DO_C:
    GRID_DIR.mkdir(parents=True, exist_ok=True)
    GRID_FULL = HAS_GPU or MOCK
    GRID_Q = GRID_Q_FULL if GRID_FULL else GRID_Q_LITE
    CONDS = [(s, q) for s in GRID_SIZES for q in GRID_Q]
    yt_all = y_all[IDX['test']]
    keep = np.ones(len(yt_all), bool) if GRID_FULL else (yt_all == OTHER)       # without a GPU: only the 'other' test images (enough for other->disease)
    T_IDX, T_Y = IDX['test'][keep], yt_all[keep]
    SETS = {'test': [np.asarray(XR[i]) for i in T_IDX], **{k: list(v) for k, v in FIELD.items()}}
    need_keys = [f'{sn}__{cond_name(s, q)}' for sn in SETS for s, q in CONDS]
    if not GRID_FULL: print('no GPU: reduced part C, q in {none, 70} and only the other test images (S4 needs a GPU)')

    models, done = {}, {}
    for a in ARMS + GRID_EXTRA_ARMS:
        for s in SEEDS:
            tag, f = f'{a}__s{s}', GRID_DIR / f'{a}__s{s}.npz'
            if f.exists():
                z = np.load(f)
                if all(k in z.files for k in need_keys) and np.array_equal(z['test_idx'], T_IDX): done[tag] = f; continue
            p = ckpt_path(a, s)
            if p is None: continue
            m = make_model(); m.load_state_dict(torch.load(p, map_location=dev)); m.eval(); models[tag] = m
    print(f'part C: {len(models)} models to evaluate, {len(done)} done | {len(CONDS)} conditions x sets {{{", ".join(f"{k}: {len(v)}" for k, v in SETS.items())}}}')
    _miss = [f'{a}__s{s}' for a in ARMS for s in SEEDS if f'{a}__s{s}' not in models and f'{a}__s{s}' not in done]
    if _miss: print('   no checkpoint (skipped):', _miss)

    if models:
        t0 = time.time(); res = {t: {} for t in models}; checked = False
        for ci, (s, q) in enumerate(CONDS):
            for sn, arrs in SETS.items():
                va = [apply_grid(a, s, q) for a in arrs]
                pr = preds_multi(models, va)
                if not checked:                                  # preds_multi must agree with logits_of (evaluation path of 03/08/13)
                    t_ = next(iter(models)); ref = logits_of(models[t_], va).argmax(1).numpy()
                    assert (ref != pr[t_]).mean() <= 0.01, 'preds_multi disagrees with logits_of'
                    checked = True
                for t in models: res[t][f'{sn}__{cond_name(s, q)}'] = pr[t]
            if (ci + 1) % 5 == 0 or ci + 1 == len(CONDS): print(f'  condition {ci+1}/{len(CONDS)} ({time.time()-t0:.0f}s)')
        for t in models:
            np.savez_compressed(GRID_DIR / f'{t}.npz', test_idx=T_IDX, test_y=T_Y, **res[t])
        del models
        if HAS_GPU: torch.cuda.empty_cache()

    # cross-check with part B: condition s112_q70 must give the same other->disease as level '112px' in the model's robustness.csv
    for f in sorted(GRID_DIR.glob('*.npz')):
        rj = OUT_DIR / f.stem / 'result.json'
        if not rj.exists(): continue
        z = np.load(f); o = z['test_y'] == OTHER
        if 'test__s112_q70' not in z.files: continue
        g = 100 * float((z['test__s112_q70'][o] != OTHER).mean())
        b = next(d['other_as_disease'] for d in json.loads(rj.read_text())['robustness'] if d['variant'] == '112px')
        if abs(g - b) > 2.7: print(f'WARNING: {f.stem}: part C gives {g:.1f}% but part B {b:.1f}% at 112 px; these should agree')
    print('part C done | result files:', len(list(GRID_DIR.glob('*.npz'))))

In [ ]:
# 7. Summary and verdict against the pre-specified criteria (rerunnable without a GPU)
def load_results():
    rows = []
    for p in sorted(OUT_DIR.glob('*__s*/result.json')):
        j = json.load(open(p))
        for d in j['robustness']:
            rows.append(dict(arm=j['arm'], seed=j['seed'], source=j.get('source'), **d))
    return pd.DataFrame(rows)

def cluster_boot(d, grp, B):
    """d = per-image values (already averaged over seeds). Resample image groups -> (mean %, lower, upper)."""
    gs = pd.Series(np.asarray(d, float)).groupby(np.asarray(grp)).agg(['sum', 'count']).values
    rng = np.random.default_rng(0); idx = rng.integers(0, len(gs), (B, len(gs)))
    bs = 100 * gs[idx, 0].sum(1) / gs[idx, 1].sum(1)
    return dict(diff=100 * float(np.mean(d)), lo=float(np.percentile(bs, 2.5)), hi=float(np.percentile(bs, 97.5)), n_images=len(d), n_groups=len(gs))

def _probs(arm, s): return OUT_DIR / f'{arm}__s{s}/test_probs.npz'
def _indicator(z, variant, metric):
    y, pred = z['y'], z[variant].argmax(1)
    if metric == 'other_as_disease':
        m = y == OTHER; return m, (pred[m] != OTHER).astype(float)
    m = y != OTHER; return m, (pred[m] == OTHER).astype(float)          # disease_as_other

def boot_diff(arm, ref, variant, metric='other_as_disease', B=N_BOOT):
    """(arm - ref) on test, paired by seed, cluster bootstrap over image groups (as in 13)."""
    seeds = [s for s in SEEDS if _probs(arm, s).exists() and _probs(ref, s).exists()]
    if not seeds: return None
    d = 0
    for s in seeds:
        m, ia = _indicator(np.load(_probs(arm, s)), variant, metric); _, ir = _indicator(np.load(_probs(ref, s)), variant, metric)
        d = d + (ia - ir)
    return dict(n_seeds=len(seeds), **cluster_boot(d / len(seeds), SPLIT['test']['group'].astype(str).values[m], B))

def boot_closure(arm, variant, B=N_BOOT):
    """CI of the share of the gap closed, (base - arm)/(base - qaug). Descriptive, not used for the verdict."""
    seeds = [s for s in SEEDS if all(_probs(a, s).exists() for a in ('base', 'qaug', arm))]
    if not seeds: return (np.nan, np.nan)
    e = {}
    for a in ('base', 'qaug', arm):
        e[a] = np.mean([_indicator(np.load(_probs(a, s)), variant, 'other_as_disease')[1] for s in seeds], 0)
    grp = SPLIT['test']['group'].astype(str).values[SPLIT['test']['y'].values == OTHER]
    g = pd.DataFrame(e).groupby(grp).sum().values                    # columns: base, qaug, arm
    rng = np.random.default_rng(0); idx = rng.integers(0, len(g), (B, len(g))); s_ = g[idx].sum(1)
    with np.errstate(divide='ignore', invalid='ignore'):
        cl = 100 * (s_[:, 0] - s_[:, 2]) / (s_[:, 0] - s_[:, 1])
    cl = cl[np.isfinite(cl)]
    return (float(np.percentile(cl, 2.5)), float(np.percentile(cl, 97.5))) if len(cl) else (np.nan, np.nan)

VERDICT = dict(part_A=A_RESULT, main='no part B results', P1=None, P2=None, P3=None, G=None, S1='not tested', S2='not tested', S3='not tested', S4='not tested')
res = load_results()
LV = CRIT['level']
if len(res):
    res.to_csv(OUT_DIR / 'jpeg14_runs.csv', index=False)
    agg = res.groupby(['arm', 'variant'])[['acc', 'acc_disease', 'other_as_disease', 'disease_as_other', 'ece']].agg(['mean', 'std', 'count'])
    agg.to_csv(OUT_DIR / 'jpeg14_summary.csv')
    show = ['original', 'jpeg_only', '224px', '160px', '112px', '80px', '160px_nojpeg', '112px_nojpeg', '80px_nojpeg']
    arms_done = [a for a in ARMS if a in res.arm.unique()]
    nseed = res.groupby('arm').seed.nunique().to_dict()
    for m, title in [('other_as_disease', 'other->disease (%)'), ('disease_as_other', 'disease->other (%)'), ('acc_disease', 'accuracy on disease images (%)')]:
        t = res.pivot_table(index='arm', columns='variant', values=m, aggfunc=['mean', 'std'])
        tab = pd.DataFrame({v: [f'{t["mean"][v][a]:.1f}±{0 if np.isnan(t["std"][v][a]) else t["std"][v][a]:.1f}' for a in arms_done] for v in show}, index=arms_done)
        print(f'\n{title}, mean±SD over seeds (n seeds = {nseed})'); print(tab.to_string())
        tab.to_csv(OUT_DIR / f'table_jpeg14_{m}.csv')

    mean = res.groupby(['arm', 'variant'])[['other_as_disease', 'disease_as_other', 'acc_disease']].mean()
    verdict = []
    for lv in ['112px', '80px']:
        if ('base', lv) not in mean.index: continue
        b = mean.loc[('base', lv), 'other_as_disease']
        gap = b - mean.loc[('qaug', lv), 'other_as_disease'] if ('qaug', lv) in mean.index else np.nan
        for a in arms_done:
            if a == 'base': continue
            bd = boot_diff(a, 'base', lv) or {}
            v = mean.loc[(a, lv), 'other_as_disease']
            clo, chi = boot_closure(a, lv) if a != 'qaug' else (np.nan, np.nan)
            verdict.append(dict(level=lv, arm=a, base=b, arm_value=v, diff=bd.get('diff'), ci_lo=bd.get('lo'), ci_hi=bd.get('hi'),
                                gap_base_minus_qaug=gap, gap_closed_pct=100 * (b - v) / gap if gap and gap > 0 else np.nan, gap_closed_ci_lo=clo, gap_closed_ci_hi=chi,
                                d_acc_disease=mean.loc[(a, lv), 'acc_disease'] - mean.loc[('base', lv), 'acc_disease'],
                                d_disease_as_other=mean.loc[(a, lv), 'disease_as_other'] - mean.loc[('base', lv), 'disease_as_other'], n_seeds=bd.get('n_seeds')))
    verdict = pd.DataFrame(verdict); verdict.to_csv(OUT_DIR / 'jpeg14_verdict.csv', index=False)
    print('\nDifference from base (arm - base, points, 95% cluster-bootstrap CI). gap_closed = share of the base - qaug gap closed:'); print(verdict.round(2).to_string(index=False))

    def get(a, col, lv=LV):
        r = verdict[(verdict.arm == a) & (verdict.level == lv)] if len(verdict) else verdict
        return float(r[col].iloc[0]) if len(r) and pd.notna(r[col].iloc[0]) else None

    # ---------- primary criteria ----------
    print(f'\n===== Primary criteria ({LV} + JPEG q{JPEG_Q}) =====')
    gap = get('qaug', 'gap_base_minus_qaug')
    have = all(nseed.get(a, 0) >= CRIT['G_min_seeds'] for a in MAIN_ARMS)
    G = bool(have and gap is not None and gap >= CRIT['G_min_gap'])
    print(f"G validity: seeds per arm {{{', '.join(f'{a}: {nseed.get(a, 0)}' for a in MAIN_ARMS)}}} | gap base - qaug = {gap if gap is None else round(gap, 1)} pts (needs >= {CRIT['G_min_gap']}) -> " + ('met' if G else 'not met / incomplete'))
    cj, hj, cr, cn = get('qaug_jpeg', 'gap_closed_pct'), get('qaug_jpeg', 'ci_hi'), get('qaug_resize', 'gap_closed_pct'), get('qaug_noise', 'gap_closed_pct')
    P1 = None if cj is None or hj is None else bool(cj >= CRIT['P1_jpeg_close_min'] and hj < 0)
    P2 = None if cr is None else bool(cr < CRIT['P2_resize_close_max'])
    P3 = None if cn is None else bool(cn < CRIT['P3_noise_close_max'])
    fmt = lambda x: 'no result' if x is None else ('met' if x else 'not met')
    if cj is not None: print(f"P1 qaug_jpeg closed {cj:.0f}% of the gap (needs >= {CRIT['P1_jpeg_close_min']:.0f}%) | upper CI of (qaug_jpeg - base) = {hj:.1f} (needs < 0) -> {fmt(P1)}")
    if cr is not None: print(f"P2 qaug_resize closed {cr:.0f}% of the gap (needs < {CRIT['P2_resize_close_max']:.0f}%) -> {fmt(P2)}")
    if cn is not None: print(f"P3 qaug_noise closed {cn:.0f}% of the gap (needs < {CRIT['P3_noise_close_max']:.0f}%) -> {fmt(P3)}")
    jr = boot_diff('qaug_jpeg', 'qaug_resize', LV)
    if not have: main = 'undecidable (not all arms/seeds complete)'
    elif not G: main = 'undecidable (validity condition G not met: base - qaug gap too small)'
    elif None in (P1, P2, P3): main = 'undecidable (incomplete)'
    elif P1 and P2 and P3: main = 'confirmed'
    elif cj >= CRIT['partial_jpeg_close_min'] and hj < 0 and cj > max(cr, cn) and jr is not None and jr['hi'] < 0: main = 'partially confirmed'
    else: main = 'not confirmed'
    if jr: print(f"also: qaug_jpeg - qaug_resize = {jr['diff']:.1f} pts [{jr['lo']:.1f}, {jr['hi']:.1f}]")
    print(f'>>> primary verdict (H-JPEG): {main}')
    VERDICT.update(main=main, P1=P1, P2=P2, P3=P3, G=G, gap=gap, closed=dict(qaug_jpeg=cj, qaug_resize=cr, qaug_noise=cn), jpeg_minus_resize=jr)

    # ---------- secondary criteria S1, S2 ----------
    print('\n===== Secondary criteria =====')
    if nseed.get('jpeg_other', 0) >= CRIT['G_min_seeds'] and G:
        ca = get('jpeg_other', 'gap_closed_pct'); b_ = boot_diff('jpeg_other', 'base', 'original'); c_ = boot_diff('jpeg_other', 'base', LV, 'disease_as_other')
        s1 = dict(a=bool(ca >= CRIT['S1a_close_min']), b=bool(b_['diff'] >= CRIT['S1b_clean_rise_min'] and b_['lo'] > 0), c=bool(c_['diff'] >= CRIT['S1c_dis_as_other_rise_min'] and c_['lo'] > 0))
        print(f"S1 jpeg_other: (a) gap closed {ca:.0f}% -> {fmt(s1['a'])} | (b) other->disease on originals vs base {b_['diff']:+.1f} pts [{b_['lo']:.1f}, {b_['hi']:.1f}] -> {fmt(s1['b'])} | "
              f"(c) disease->other at {LV} vs base {c_['diff']:+.1f} pts [{c_['lo']:.1f}, {c_['hi']:.1f}] -> {fmt(s1['c'])}")
        VERDICT['S1'] = 'met' if all(s1.values()) else 'not met'; VERDICT['S1_detail'] = dict(closed=ca, clean_rise=b_, disease_as_other_rise=c_, **s1)
    else: print('S1 jpeg_other: not tested')
    if nseed.get('jpeg_disease', 0) >= CRIT['G_min_seeds'] and G:
        cd = get('jpeg_disease', 'gap_closed_pct')
        VERDICT['S2'] = 'met' if cd < CRIT['S2_close_max'] else 'not met'; VERDICT['S2_detail'] = dict(closed=cd)
        print(f"S2 jpeg_disease closed {cd:.0f}% of the gap (needs < {CRIT['S2_close_max']:.0f}%) -> {VERDICT['S2']}")
    else: print('S2 jpeg_disease: not tested')

    print('\nresult source:', res.drop_duplicates(['arm', 'seed']).groupby('arm').source.agg(lambda x: '/'.join(sorted(set(map(str, x))))).to_dict())
    for a, ref, tol in [('base', 43.7, 10), ('qaug', 9.5, 6)]:
        if (a, '112px') in mean.index and not MOCK and abs(mean.loc[(a, '112px'), 'other_as_disease'] - ref) > tol:
            print(f'WARNING: {a} @112 = {mean.loc[(a, "112px"), "other_as_disease"]:.1f}% differs from 13 ({ref}%) by more than {tol} pts. Check checkpoints and resampling.')

    # figure: part B arms over the standard levels
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    for a in arms_done:
        for k, m in enumerate(['other_as_disease', 'acc_disease']):
            mu = [agg.loc[(a, v), (m, 'mean')] for v in STD_LEVELS]; sd = [np.nan_to_num(agg.loc[(a, v), (m, 'std')]) for v in STD_LEVELS]
            ax[k].errorbar(range(len(STD_LEVELS)), mu, yerr=sd, marker='o', capsize=2, label=a, lw=2 if a in ('base', 'qaug_jpeg', 'qaug') else 1.2, ls='--' if a in OPT_ARMS else '-')
    for k, (yl, tt) in enumerate([('non-durian images predicted as disease (%)', '(a) Rejection failure'), ('accuracy on disease images (%)', '(b) Disease recognition')]):
        ax[k].set_xticks(range(len(STD_LEVELS))); ax[k].set_xticklabels(STD_LEVELS); ax[k].set(ylabel=yl, title=tt, xlabel=f'test image: reduced size + JPEG q{JPEG_Q}')
    ax[0].legend(fontsize=8); plt.tight_layout()
    for ext in ['png', 'pdf']: fig.savefig(OUT_DIR / f'fig_jpeg14_components.{ext}', dpi=300)
    plt.show() if (IN_COLAB or IN_KAGGLE) else plt.close(fig)
else:
    print('no part B results yet')

# ---------- Part C: quality grid, S3 and S4 ----------
GRID_DIR = OUT_DIR / 'grid'
QLAB = lambda q: 'none' if q is None else str(q)
GR, grows = {}, []
for f in sorted(GRID_DIR.glob('*.npz')) if GRID_DIR.exists() else []:
    arm, seed = f.stem.rsplit('__s', 1); z = np.load(f); GR[(arm, int(seed))] = z
    o = z['test_y'] == OTHER
    for k in z.files:
        if '__' not in k: continue
        sn, cond = k.split('__', 1); s, q = parse_cond(cond); p = z[k]
        if sn == 'test':
            d = dict(n=int(o.sum()), other_as_disease=100 * float((p[o] != OTHER).mean()))
            if (~o).any(): d.update(disease_as_other=100 * float((p[~o] == OTHER).mean()), acc_disease=100 * float((p[~o] == z['test_y'][~o]).mean()))
        else:
            d = dict(n=len(p), other_as_disease=100 * float((p != OTHER).mean()))
        grows.append(dict(arm=arm, seed=int(seed), set=sn, size=s, q=QLAB(q), **d))
gdf = pd.DataFrame(grows)

def grid_diff(arm, sn, c1, c2, B, block=None):
    """(condition c1 - c2) of other->disease, paired per image, averaged over seeds. Test set: clusters of image groups; orchard set: per image or blocks of consecutive photographs."""
    zs = [z for (a, s), z in GR.items() if a == arm and f'{sn}__{c1}' in z.files and f'{sn}__{c2}' in z.files]
    if not zs: return None
    d = 0
    for z in zs:
        m = (z['test_y'] == OTHER) if sn == 'test' else np.ones(len(z[f'{sn}__{c1}']), bool)
        d = d + ((z[f'{sn}__{c1}'][m] != OTHER).astype(float) - (z[f'{sn}__{c2}'][m] != OTHER).astype(float))
    d = d / len(zs)
    grp = np.array([ROW_GROUP[r] for r in zs[0]['test_idx'][m]]) if sn == 'test' else (np.arange(len(d)) // block if block else np.arange(len(d)))
    return dict(n_seeds=len(zs), **cluster_boot(d, grp, B))

if len(gdf):
    gdf.to_csv(OUT_DIR / 'grid_runs.csv', index=False)
    gs = gdf.groupby(['arm', 'set', 'size', 'q'])['other_as_disease'].agg(['mean', 'std', 'count']).reset_index()
    gs.to_csv(OUT_DIR / 'grid_summary.csv', index=False)
    qcols = [QLAB(q) for q in GRID_Q_FULL if QLAB(q) in set(gdf.q)]
    for a in [a for a in ['base', 'qaug_jpeg', 'qaug_resize', 'qaug_noise', 'qaug'] if a in set(gdf.arm)]:
        t = gs[(gs.arm == a) & (gs.set == 'test')].pivot(index='size', columns='q', values='mean').reindex(index=sorted(set(gdf['size']), reverse=True), columns=qcols)
        t.to_csv(OUT_DIR / f'table_grid_{a}.csv')
        if a in ('base', 'qaug_jpeg', 'qaug'): print(f'\npart C, {a}: other->disease (%) on test, mean over seeds | rows = size px, columns = JPEG quality (none = uncompressed)'); print(t.round(1).to_string())

    # S4: effect size by compression level
    a4, s4 = CRIT['S4_arm'], CRIT['S4_size']
    r4 = gs[(gs.arm == a4) & (gs.set == 'test') & (gs['size'] == s4)].set_index('q')['mean']
    qs = [q for q in GRID_Q_FULL if q is not None]
    n4 = gdf[(gdf.arm == a4) & (gdf.set == 'test')].seed.nunique()
    if all(str(q) in r4.index for q in qs) and 'none' in r4.index and n4 >= CRIT['G_min_seeds']:
        d4 = grid_diff(a4, 'test', cond_name(s4, 30), cond_name(s4, 95), N_BOOT)
        rho4 = float(spearmanr(qs, [r4[str(q)] for q in qs]).statistic); light = float(r4['95'] - r4['none'])
        s4r = dict(a=bool(d4['diff'] >= CRIT['S4_min_range'] and d4['lo'] > 0), b=bool(rho4 <= CRIT['S4_max_spearman']), c=bool(light < CRIT['S4_max_q95_minus_none']))
        VERDICT['S4'] = 'met' if all(s4r.values()) else 'not met'; VERDICT['S4_detail'] = dict(range_q30_minus_q95=d4, spearman=rho4, q95_minus_none=light, **s4r)
        print(f"\nS4 ({a4}, {s4} px): (a) q30 - q95 = {d4['diff']:.1f} pts [{d4['lo']:.1f}, {d4['hi']:.1f}] | (b) Spearman(q, other->disease) = {rho4:.2f} | (c) q95 - uncompressed = {light:.1f} pts -> {VERDICT['S4']}")
    else: print('\nS4: not tested (needs the full grid for base on all seeds; run part C on a GPU)')

    # S3 and orchard table: JPEG versus resize only
    frows = []
    for sn in [s for s in sorted(set(gdf.set)) if s != 'test']:
        for a in [a for a in ['base', 'qaug', 'qaug_jpeg', 'qaug_resize'] if a in set(gdf.arm)]:
            for s in [s for s in GRID_SIZES if s < 256]:
                c1, c2 = cond_name(s, 70), cond_name(s, None)
                d = grid_diff(a, sn, c1, c2, N_BOOT_FIELD); db = grid_diff(a, sn, c1, c2, N_BOOT_FIELD, block=5)
                if d is None: continue
                g_ = gs[(gs.arm == a) & (gs.set == sn) & (gs['size'] == s)].set_index('q')['mean']
                frows.append(dict(set=sn, arm=a, size=s, n_images=d['n_images'], n_seeds=d['n_seeds'], jpeg_q70=g_.get('70'), resize_only=g_.get('none'),
                                  original=float(gs[(gs.arm == a) & (gs.set == sn) & (gs['size'] == 256) & (gs.q == 'none')]['mean'].iloc[0]),
                                  diff=d['diff'], ci_lo=d['lo'], ci_hi=d['hi'], block5_lo=db['lo'], block5_hi=db['hi']))
    fdf = pd.DataFrame(frows)
    if len(fdf):
        fdf.to_csv(OUT_DIR / 'field_jpeg_vs_resize.csv', index=False)
        print('\nOrchard photographs (non-durian): labelled as disease (%), JPEG q70 vs resize only at the same size (diff = JPEG - resize only):'); print(fdf.round(1).to_string(index=False))
        r3 = fdf[(fdf.set == CRIT['S3_set']) & (fdf.arm == CRIT['S3_arm']) & (fdf['size'] == 112)]
        if len(r3) and r3.n_seeds.iloc[0] >= CRIT['G_min_seeds']:
            r3 = r3.iloc[0]
            VERDICT['S3'] = 'met' if (r3['diff'] >= CRIT['S3_min_diff'] and r3['ci_lo'] > 0) else 'not met'; VERDICT['S3_detail'] = {k: (v if isinstance(v, str) else float(v)) for k, v in r3.to_dict().items()}
            print(f"S3 ({CRIT['S3_arm']}, {CRIT['S3_set']}, n={int(r3['n_images'])}): 112 px + JPEG {r3['jpeg_q70']:.1f}% - resize only {r3['resize_only']:.1f}% = {r3['diff']:.1f} pts [{r3['ci_lo']:.1f}, {r3['ci_hi']:.1f}] (blocks of 5 [{r3['block5_lo']:.1f}, {r3['block5_hi']:.1f}]) -> {VERDICT['S3']}")
    if VERDICT['S3'] == 'not tested': print(f"\nS3: not tested (no orchard set {CRIT['S3_set']}, or checkpoints of {CRIT['S3_arm']} incomplete)")

    # exploratory: which classes misclassified 'other' images go to (base, all seeds), and whether this follows part A's per-class artefacts
    drows = []
    for (a, sd), z in GR.items():
        if a != 'base': continue
        o = z['test_y'] == OTHER
        for k in z.files:
            if not k.startswith('test__'): continue
            p = z[k][o]; p = p[p != OTHER]
            for c_, n_ in zip(*np.unique(p, return_counts=True)): drows.append(dict(cond=k.split('__', 1)[1], label=CLASSES[int(c_)], n=int(n_)))
    if drows:
        dd = pd.DataFrame(drows).groupby(['cond', 'label']).n.sum().reset_index()
        dd['share_pct'] = 100 * dd.n / dd.groupby('cond').n.transform('sum'); dd.to_csv(OUT_DIR / 'grid_error_destinations.csv', index=False)
        k112 = dd[dd.cond == cond_name(112, 70)].sort_values('share_pct', ascending=False)
        print(f'\n(exploratory) base @112 px + JPEG q70: {int(k112.n.sum())} misclassified other images went to:', {r.label: round(r.share_pct, 1) for r in k112.itertuples()})
        if RUN_PART_A and len(k112) >= 2:
            mm = by_cls.set_index('label').lattice_median.to_frame().join(k112.set_index('label').share_pct).fillna({'share_pct': 0}).drop(index='other', errors='ignore')
            rr = spearmanr(mm.lattice_median, mm.share_pct)
            print(f'(exploratory) Spearman between per-class block artefacts (part A) and share of misclassified images received = {rr.statistic:.2f} (p = {rr.pvalue:.2f}, n = {len(mm)} classes)')
            VERDICT['explore_class_trace_vs_destination'] = dict(spearman=float(rr.statistic), p=float(rr.pvalue), n=len(mm))

    # figure: grid curves
    pa = [a for a in ['base', 'qaug_resize', 'qaug_jpeg', 'qaug'] if a in set(gdf.arm)]
    sizes = sorted(set(gdf['size']), reverse=True)
    fig, ax = plt.subplots(1, len(pa), figsize=(3.6 * len(pa) + 0.6, 3.8), sharey=True, squeeze=False); ax = ax[0]
    cm = plt.get_cmap('viridis')
    for k, a in enumerate(pa):
        for i, q in enumerate(qcols):
            g_ = gs[(gs.arm == a) & (gs.set == 'test') & (gs.q == q)].set_index('size').reindex(sizes)
            ax[k].errorbar(range(len(sizes)), g_['mean'], yerr=g_['std'].fillna(0), marker='o', ms=3, capsize=2, lw=2 if q == 'none' else 1.2,
                           color='k' if q == 'none' else cm(i / max(1, len(qcols) - 1)), ls='--' if q == 'none' else '-', label='no JPEG' if q == 'none' else f'JPEG q{q}')
        ax[k].set_xticks(range(len(sizes))); ax[k].set_xticklabels(sizes); ax[k].set(title=a, xlabel='image size before enlargement (px)')
    ax[0].set_ylabel('non-durian images predicted as disease (%)'); ax[-1].legend(fontsize=7); plt.tight_layout()
    for ext in ['png', 'pdf']: fig.savefig(OUT_DIR / f'fig_jpeg14_grid.{ext}', dpi=300)
    plt.show() if (IN_COLAB or IN_KAGGLE) else plt.close(fig)
else:
    print('\nno part C results yet')

# ---------- verdict summary ----------
def _clean(o):
    if isinstance(o, dict): return {k: _clean(v) for k, v in o.items()}
    if isinstance(o, (np.floating, np.integer, np.bool_)): return o.item()
    return o
(OUT_DIR / 'jpeg14_verdict.json').write_text(json.dumps(_clean(VERDICT), indent=1, ensure_ascii=False))
print('\n' + '=' * 70)
print('Summary against the pre-specified criteria' + (' (synthetic data; numbers are meaningless)' if MOCK else ''))
print(f"  primary H-JPEG (part B)     : {VERDICT['main']}   [P1 {fmt(VERDICT['P1']) if len(res) else '-'} | P2 {fmt(VERDICT['P2']) if len(res) else '-'} | P3 {fmt(VERDICT['P3']) if len(res) else '-'}]")
print(f"  S1 reversal (jpeg_other)    : {VERDICT['S1']}")
print(f"  S2 jpeg_disease no repair   : {VERDICT['S2']}")
print(f"  S3 orchard photographs      : {VERDICT['S3']}")
print(f"  S4 compression dose-response: {VERDICT['S4']}")
print(f"  part A artefacts by source  : {A_RESULT.get('wording', A_RESULT['status'])}")
if len(res) and res.groupby('arm').seed.nunique().min() < 3 and not MOCK: print('WARNING: not all arms have 3 seeds; do not report these numbers')
print('=' * 70)

In [ ]:
# 8. Bundle results (without best.pt)
ZIP = OUT_DIR.parent / 'jpeg14_results.zip'
with zipfile.ZipFile(ZIP, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT_DIR.rglob('*')):
        if p.is_file() and p.suffix != '.pt':
            z.write(p, Path('jpeg14') / p.relative_to(OUT_DIR))
print('saved', ZIP, f'({ZIP.stat().st_size/1e6:.1f} MB)')